# Correcting a classification raster inside a polygon

Take a closed vector polygon, a categorical raster (LANDFIRE EVT here), and a list of the
raster's class values that are **known to be wrong over that polygon**. Produce a corrected
raster in which those pixels — and only those pixels — carry the class the satellite
embeddings say they actually are.

The case this was built for: a stand is clearcut, LANDFIRE's source imagery sees bare
ground, and the pixel is labelled *Southeastern Ruderal Grassland* years after it went back
to pine. The polygon asserts where the labelling is suspect; the eligible class list asserts
which labels there may be overwritten; everything else in the neighbourhood is evidence.

## Method

1. **Look first.** NAIP imagery for ±10 years around the target year, one mosaic per year on
   a slider, with the polygon drawn over it as a transparent fill with a hatched border.
   Before any model runs, you confirm with your own eyes that the ground inside that
   boundary is not what the raster says it is.
2. **Split the neighbourhood.** Read the raster over the polygon plus a padding ring. Every
   pixel whose class is *not* eligible is a trusted label — inside the polygon and outside
   it. The eligible pixels inside the polygon are the apply set.
3. **Attach embeddings.** AlphaEarth annual satellite embeddings (64 dimensions, 10 m) at
   every sampled pixel, for the raster's own vintage.
4. **Fit and score honestly.** Multinomial logistic regression, cross-validated on
   *spatially blocked* folds, against a label-shuffle baseline. Neighbouring 30 m pixels are
   not independent samples and a random split will tell you so in the most flattering
   possible way.
5. **Correct, with a confidence floor.** A prediction replaces the original value only when
   it differs from it *and* clears the confidence threshold. Everything else keeps the label
   it came with — a pixel that stays suspect is recoverable, a confidently wrong correction
   is not.

**Why the trusted set includes the polygon's own pixels.** An AOI usually sits in a
landscape it does not resemble. Training only outside makes the model extrapolate across the
boundary; the undisputed pixels inside are the closest thing to in-situ reference data
available.

## Outputs

Written to `data/interim/raster_correction/<slug>/`:

| File | Contents |
|---|---|
| `<slug>_corrected.tif` | One band, source dtype and nodata — a drop-in replacement for the clipped input |
| `<slug>_diagnostics.tif` | Three float32 bands: original class, per-pixel confidence, changed flag |
| `<slug>_manifest.json` | Parameters, sample counts, CV report, every from→to transition with its count |

## Prerequisites

- **Earth Engine**, authenticated *and* pointed at a Cloud project:

  ```bash
  uv run earthengine authenticate
  ```

  The project is `EE_PROJECT` in §2 (`perseus-gee`); set it to `None` to use a machine
  default from `earthengine set_project`. A stored credential does not carry a project — a
  valid token with none fails with "no project found", and re-authenticating cannot fix it.
- **The classification raster**, as a Florida clip (75 MB, not 2.99 GB):

  ```bash
  uv run python -m pipeline.raster_clip \
      --raster raw.landfire.evt_tif --region config/extent.geojson --name LF2022_EVT_FL
  ```

  §6 falls back to the CONUS raster if the clip is absent.
- **A closed polygon.** A vector file, or the lon/lat bounding box fallback in §2.

Logic lives in `pipeline/s5_imagery/{raster_correction,feature_sources,naip_viewer}.py` and is
unit-tested offline in `tests/test_s5_{raster_correction,feature_sources,naip_viewer}.py`. This
notebook is the interactive interface to it, not the implementation.

> **Commit gotcha.** Executing the map cell embeds widget state and can grow this file by
> many megabytes. Clear outputs, or strip `nb.metadata["widgets"]`, before committing.

## 1. Setup

In [ ]:
import json
import logging
import sys
from pathlib import Path

import numpy as np
import pandas as pd


def repo_root() -> Path:
    """The repository root, found from wherever Jupyter was started."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "pipeline").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from inside the repository (see notebooks/README.md)")


REPO = repo_root()
for path in (REPO, REPO / "notebooks"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import clearcut_ag_common as cac  # noqa: E402  — EVT paths, class table, Earth Engine init
from pipeline.s5_imagery import feature_sources, naip_acquire, naip_viewer, vectors  # noqa: E402
from pipeline.s5_imagery import raster_correction as rc  # noqa: E402

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s", force=True)

print("repo:", REPO)

## 2. Parameters

Everything adjustable lives here. The defaults describe the LANDFIRE case: the LF2022 EVT
raster, the three EVT classes this project has already shown are confused with recent
clearcuts (`notes/clearcut-vs-agriculture-embeddings.md`), and a 2022 target year matching
both the EVT vintage and TreeMap 2022.

In [ ]:
# -- what is being corrected ---------------------------------------------------------------
RASTER_PATH = None                              # None -> the Florida EVT clip, else CONUS EVT
RASTER_BAND = 1
ELIGIBLE_CLASSES = list(cac.CONFUSED_EVT_VALUES)  # 7997 pasture/hay, 9823 ruderal grass, 9585 floodplain shrub

# LANDFIRE codes ocean and out-of-CONUS ground as -9999 *inside* the valid range — the
# GeoTIFF nodata (32767) does not cover it. Left in, that fill becomes a trusted class and
# a coastal AOI trains the classifier on what the ocean looks like.
INVALID_CLASS_VALUES = (-9999,)

# -- where ---------------------------------------------------------------------------------
AOI_PATH = None            # GeoJSON / GeoPackage / shapefile holding the closed polygon
AOI_LAYER = None           # layer name, for multi-layer sources
AOI_BBOX = (-82.62, 30.10, -82.60, 30.12)   # lon/lat fallback used when AOI_PATH is None

# -- when ----------------------------------------------------------------------------------
TARGET_YEAR = 2022         # the raster's vintage; also the embedding year
WINDOW_BACK_YEARS = 10
WINDOW_FORWARD_YEARS = 10  # trimmed to what exists — see the warnings in §4

# -- how -----------------------------------------------------------------------------------
PAD_M = 500.0              # context ring around the polygon; the outside training evidence
MIN_CONFIDENCE = 0.60      # below this, the original label stands
MAX_PER_CLASS = 400        # training samples per class
MIN_PER_CLASS = 30         # rarer classes are dropped from the vocabulary, not fitted
BLOCK_M = 300.0            # spatial CV block edge (10 pixels of a 30 m raster)
N_SPLITS = 5
SEED = 0

# -- Earth Engine --------------------------------------------------------------------------
# Earth Engine requires a Cloud project, and a stored credential does not carry one: a valid
# token with no project fails with "no project found", and re-authenticating cannot fix it.
# Set None to fall back to the machine default (`uv run earthengine set_project <id>`).
EE_PROJECT = "perseus-gee"

# -- outputs -------------------------------------------------------------------------------
RUN_SLUG = "aoi_correction"
OUT_DIR = REPO / "data" / "interim" / "raster_correction" / RUN_SLUG

## 3. The polygon

The polygon is the whole claim: *the classification is wrong in here, and nowhere else*.
Load it, or fall back to the bounding box above for a first run.

An alternative worth knowing about: the geemap map in §5 has a draw control, so a polygon
sketched on the imagery is available afterwards as `slider.map.user_roi`. Set `AOI_PATH` or
`AOI_BBOX` for a reproducible run; draw for exploration.

In [ ]:
from shapely.geometry import box  # noqa: E402

if AOI_PATH:
    aoi = vectors.layer_geometry(vectors.load_layer(AOI_PATH, AOI_LAYER))
    aoi_source = str(AOI_PATH)
else:
    aoi = box(*AOI_BBOX)
    aoi_source = f"AOI_BBOX {AOI_BBOX}"

print(f"AOI: {aoi_source}")
print(f"  geometry {aoi.geom_type}, {vectors.area_ha(aoi):,.1f} ha")
print(f"  bounds   {[round(v, 5) for v in aoi.bounds]} (lon/lat)")

## 4. NAIP years around the target

A ±10-year window around 2022 asks for imagery through 2032, and most of that has not been
flown. The window resolver trims to what can exist and says what it trimmed, because "four
years after the target instead of ten" changes how much weight the after-comparison carries.

In [ ]:
ee = cac.init_ee(EE_PROJECT)
extent_ee = vectors.to_ee_geometry(aoi)

available_years = naip_acquire.available_naip_years(ee, extent_ee)
print(f"NAIP years over this AOI: {available_years}")

year_window = naip_viewer.resolve_year_window(
    TARGET_YEAR,
    back_years=WINDOW_BACK_YEARS,
    forward_years=WINDOW_FORWARD_YEARS,
    available=available_years,
)

print(f"\nrequested : {year_window.requested[0]}–{year_window.requested[-1]}")
print(f"usable    : {year_window.years}")
print(f"  before {TARGET_YEAR}: {year_window.before}")
print(f"  after  {TARGET_YEAR}: {year_window.after}")
for message in year_window.warnings:
    print(f"  ! {message}")

## 5. Look at the ground before modelling it

The slider swaps the NAIP mosaic underneath the polygon. Mosaics are built on first view and
cached, so moving through the window costs Earth Engine calls once per year.

What to look for: does the inside of the hatched boundary change between the pre-target years
and the target year, and does it look like the eligible class or like the surrounding matrix?
A polygon that looks the same as its surroundings in every year is a polygon whose
classification probably is not wrong.

Coverage is reported per year, gap-filled from neighbouring years exactly as an export would
be — the mosaic on screen is the one you would get if you exported it, not a prettier one.

In [ ]:
slider = naip_viewer.NaipYearSlider(
    ee,
    aoi,
    year_window.years,
    target_year=TARGET_YEAR,
    available_years=available_years,
)
slider.widget()

## 6. Read the classification raster over the polygon

Only the polygon plus its padding ring is read, so the source size barely matters — but it
still has to be reachable. Preference order:

1. **`data/interim/clips/LF2022_EVT_FL.tif`** — Florida clipped out of the CONUS EVT,
   75 MB instead of 2.99 GB, same CRS, same 30 m grid, pixel-identical. Produce it with:

   ```bash
   uv run python -m pipeline.raster_clip        --raster raw.landfire.evt_tif --region config/extent.geojson --name LF2022_EVT_FL
   ```

2. **The full CONUS raster**, via `cac.evt2022_tif_path()`. At 2.99 GB it is over
   `pipeline.data_access`'s fetch cap, so off the workstation this raises with the one
   `rclone` command that stages it rather than pulling it silently mid-cell.

Set `RASTER_PATH` to override both.

In [ ]:
FLORIDA_EVT_CLIP = REPO / "data" / "interim" / "clips" / "LF2022_EVT_FL.tif"

if RASTER_PATH:
    raster_path = RASTER_PATH
elif FLORIDA_EVT_CLIP.exists():
    raster_path = FLORIDA_EVT_CLIP
else:
    print("No Florida clip found; falling back to the CONUS raster. See the cell above.")
    raster_path = cac.evt2022_tif_path()

evt_lookup = cac.load_evt2022_lookup(cac.evt2022_csv_path())
CLASS_NAMES = {value: record["name"] for value, record in evt_lookup.items()}

# One request object carries every parameter through the steps below, and into the manifest.
request = rc.CorrectionRequest(
    aoi=aoi,
    raster_path=raster_path,
    eligible_classes=ELIGIBLE_CLASSES,
    out_dir=OUT_DIR,
    slug=RUN_SLUG,
    aoi_source=aoi_source,
    invalid_values=INVALID_CLASS_VALUES,
    pad_m=PAD_M,
    band=RASTER_BAND,
    max_per_class=MAX_PER_CLASS,
    min_per_class=MIN_PER_CLASS,
    block_m=BLOCK_M,
    n_splits=N_SPLITS,
    min_confidence=MIN_CONFIDENCE,
    seed=SEED,
    class_names=CLASS_NAMES,
)

window = rc.read_window(raster_path, aoi, pad_m=request.pad_m, band=request.band)
pixels = rc.pixel_table(window, aoi, request.invalid_values)

print(f"raster : {raster_path}")
print(f"window : {window.shape[0]} x {window.shape[1]} px, {window.crs}")
print(f"pixels : {len(pixels):,} valid ({int(pixels['inside'].sum()):,} inside the polygon)")

### What is in there

Every class in the window, split by whether it falls inside the polygon. The eligible classes
should account for a substantial share of the inside; if they do not, the polygon is drawn
around something other than the problem.

In [ ]:
counts = (
    pixels.groupby(["class_value", "inside"])
    .size()
    .unstack("inside")
    .reindex(columns=[False, True], fill_value=0)
    .rename(columns={False: "outside_px", True: "inside_px"})
)
inventory = (
    counts.assign(
        name=[CLASS_NAMES.get(int(value), "?") for value in counts.index],
        eligible=[int(value) in set(ELIGIBLE_CLASSES) for value in counts.index],
        inside_ha=counts["inside_px"] * window.pixel_area_m2 / 10_000,
    )
    .sort_values("inside_px", ascending=False)
    .reset_index()
)
eligible_share = inventory.loc[inventory["eligible"], "inside_px"].sum() / max(int(pixels["inside"].sum()), 1)
print(f"eligible classes cover {eligible_share:.1%} of the polygon\n")
inventory.head(15)

## 7. Trusted labels and the apply set

Trusted: every non-eligible pixel, inside and outside, capped per class. Classes with too few
pixels to learn or cross-validate are dropped from the vocabulary and reported rather than
fitted — a class the model has seen nine times is a class it will assign wrongly and
confidently.

Apply: eligible pixels **inside** the polygon. Eligible pixels outside it are left alone.

In [ ]:
split = rc.split_samples(
    pixels,
    request.eligible_classes,
    max_per_class=request.max_per_class,
    min_per_class=request.min_per_class,
    seed=request.seed,
)

used = split.training["class_value"].astype(int).value_counts()
print(f"training : {len(split.training):,} px over {len(split.class_counts)} classes "
      f"(sampled from {sum(split.class_counts.values()):,} trusted px, cap {request.max_per_class}/class)")
for value, available in sorted(split.class_counts.items(), key=lambda item: -item[1]):
    print(f"    {value:>6}  {used.get(value, 0):>5,} of {available:>6,}  {CLASS_NAMES.get(value, '?')[:52]}")
if split.dropped_classes:
    print(f"  dropped (< {request.min_per_class} px): {split.dropped_classes}")
if split.absent_eligible:
    print(f"  eligible classes absent from the window: {split.absent_eligible}")

inside_from = split.training["inside"].sum()
print(f"  of which inside the polygon: {inside_from:,} ({inside_from / len(split.training):.1%})")
print(f"\napply    : {len(split.apply):,} px inside the polygon")

## 8. Attach the embeddings

AlphaEarth annual embeddings, sampled point by point through Earth Engine in chunks. This is
the slow cell: roughly one round-trip per 500 pixels.

The vintage is `TARGET_YEAR` — the raster's own. A later year would let the model see change
the raster could not have known about, which is a different (and sometimes useful) question;
whatever you choose is recorded in the manifest.

Pixels AlphaEarth cannot answer for come back as `NaN` and are dropped here rather than
imputed. A partially observed embedding is not a weaker observation of the same thing.

In [ ]:
source = feature_sources.AlphaEarthEmbeddings(year=TARGET_YEAR)
print(source.description)

training, X_train = rc.attach_features(rc.attach_lonlat(split.training, window.crs), source)
apply_rows, X_apply = rc.attach_features(rc.attach_lonlat(split.apply, window.crs), source)

y_train = training["class_value"].astype(int).to_numpy()
print(f"training features : {X_train.shape}  (dropped {len(split.training) - len(training):,} without coverage)")
print(f"apply features    : {X_apply.shape}  (dropped {len(split.apply) - len(apply_rows):,} without coverage)")

## 9. Spatially blocked cross-validation

Folds are groups of 300 m blocks, so a held-out fold is ground the model has not seen rather
than pixels adjacent to ones it has. The shuffled-label baseline runs the identical procedure
on permuted labels: **if the two accuracies are close, the embeddings carry no signal for this
landscape** and the headline number is an artifact of class imbalance, not evidence.

A small AOI can fail to produce enough blocks to fold. That is reported, not raised — the
correction can still be produced, but its accuracy is then unmeasured and the manifest says so.

In [ ]:
groups = rc.block_groups(training["x"].to_numpy(), training["y"].to_numpy(), request.block_m)
cv_report = rc.spatial_cross_validate(
    X_train, y_train, groups, n_splits=request.n_splits, seed=request.seed
)

if cv_report.skipped_reason:
    print(f"! cross-validation skipped: {cv_report.skipped_reason}")
else:
    print(f"{cv_report.n_samples:,} samples, {cv_report.n_classes} classes, "
          f"{cv_report.n_blocks} blocks, {cv_report.n_folds} folds\n")
    print(f"                accuracy   macro F1")
    print(f"  blocked CV      {cv_report.accuracy:.3f}      {cv_report.macro_f1:.3f}")
    print(f"  shuffled        {cv_report.shuffled_accuracy:.3f}      {cv_report.shuffled_macro_f1:.3f}")
    print(f"  margin          {cv_report.accuracy - cv_report.shuffled_accuracy:+.3f}\n")
    display(
        pd.DataFrame(
            [
                {"class": value, "name": CLASS_NAMES.get(value, "?")[:52], **scores}
                for value, scores in sorted(cv_report.per_class.items())
            ]
        ).sort_values("f1", ascending=False)
    )

## 10. Fit, predict, decide

The fitted model is applied to every eligible pixel inside the polygon. A prediction is
written only where it both differs from the original label and clears `MIN_CONFIDENCE`;
everything else keeps what it had.

In [ ]:
model = rc.build_model(request.seed)
model.fit(X_train, y_train)

outcome = rc.predict_corrections(
    model, apply_rows, X_apply, n_eligible=len(split.apply), min_confidence=request.min_confidence
)

print(f"eligible pixels inside the polygon : {outcome.n_eligible:,}")
print(f"  no embedding coverage            : {outcome.n_featureless:,}")
print(f"  below {request.min_confidence:.2f} confidence            : {outcome.n_low_confidence:,}")
print(f"  corrected                        : {outcome.n_changed:,} "
      f"({outcome.n_changed * window.pixel_area_m2 / 10_000:,.1f} ha)\n")

rc.transition_table(outcome, window.pixel_area_m2, CLASS_NAMES)

### Where the confidence sits

The shape of this distribution is the thing to look at, not its mean. A bimodal split — a
confident mass and an uncertain tail — is a model making distinctions. A single lump just
above the threshold is a model guessing at the same rate everywhere, and the threshold rather
than the evidence is deciding what gets corrected.

In [ ]:
import matplotlib.pyplot as plt  # noqa: E402

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.hist(outcome.applied["confidence"], bins=40, color="#4c72b0", edgecolor="white")
ax.axvline(request.min_confidence, color="#c44e52", linestyle="--",
           label=f"threshold {request.min_confidence:.2f}")
ax.set_xlabel("max class probability")
ax.set_ylabel("eligible pixels")
ax.set_title("Prediction confidence inside the polygon")
ax.legend()
plt.tight_layout()

## 11. Write the outputs

Two rasters and a manifest. They are co-registered with the window read in §6, so the
corrected file drops straight in as a clipped replacement for the input.

The whole sequence from §6 to here is also available as one call —
`rc.correct_raster(request, source)` — for reruns where nothing needs inspecting.

In [ ]:
corrected = rc.apply_to_window(window, outcome)

corrected_path = rc.write_corrected(OUT_DIR / f"{RUN_SLUG}_corrected.tif", window, corrected)
diagnostics_path = rc.write_diagnostics(
    OUT_DIR / f"{RUN_SLUG}_diagnostics.tif", window, rc.diagnostics_stack(window, outcome)
)

manifest = rc.build_manifest(request, source, window, split, cv_report, outcome, len(training))
manifest["outputs"] = {"corrected": corrected_path.name, "diagnostics": diagnostics_path.name}
manifest_path = OUT_DIR / f"{RUN_SLUG}_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2))

for path in (corrected_path, diagnostics_path, manifest_path):
    print(f"{path.stat().st_size / 1e6:8.2f} MB  {path}")

## 12. Before and after

Original, corrected, and the confidence surface, on one colour scale so a changed pixel is
visible as a colour change rather than as a difference in palette. The polygon is drawn on
each: nothing outside it may differ between the first two panels.

In [ ]:
import geopandas as gpd  # noqa: E402
from rasterio.plot import plotting_extent  # noqa: E402

aoi_local = gpd.GeoSeries([aoi], crs=vectors.WGS84).to_crs(window.crs)
extent = plotting_extent(window.values, window.transform)

palette = np.unique(np.concatenate([np.unique(window.values), np.unique(corrected)]))


def recode(array):
    """Class values to palette indices, so both panels share one colour scale."""
    return np.searchsorted(palette, array)


fig, axes = plt.subplots(1, 3, figsize=(17, 5.6))
for ax, data, title in (
    (axes[0], recode(window.values), f"original ({Path(raster_path).name})"),
    (axes[1], recode(corrected), f"corrected ({outcome.n_changed:,} px changed)"),
):
    ax.imshow(data, extent=extent, cmap="tab20", interpolation="nearest",
              vmin=0, vmax=max(len(palette) - 1, 1))
    ax.set_title(title, fontsize=11)

confidence = rc.diagnostics_stack(window, outcome)[1]
image = axes[2].imshow(confidence, extent=extent, cmap="viridis", interpolation="nearest",
                       vmin=0, vmax=1)
axes[2].set_title("confidence (eligible pixels only)", fontsize=11)
fig.colorbar(image, ax=axes[2], fraction=0.046)

for ax in axes:
    aoi_local.boundary.plot(ax=ax, color="#ffd166", linewidth=1.8)
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()

## Interpretation, limits, next steps

**Read the manifest before trusting the raster.** `cross_validation.accuracy` beside
`cross_validation.shuffled_accuracy` is the number that says whether any of this is
signal. `corrections.transitions` says what actually happened, class by class. A run with a
thin margin over the shuffled baseline has produced a corrected raster; it has not produced
evidence.

**Known limits of this v1.**

- **Any trusted class can be assigned, including implausible ones.** A live run over a
  north-Florida ruderal-grassland cluster reassigned 7 pixels to *Open Water* — the model
  separates water almost perfectly (held-out F1 0.99), so wet or shadowed grassland lands
  there confidently. Read the §10 transition table as a result to be checked, not accepted:
  a grassland-to-water "correction" is a flag, not a finding. Restricting the assignable
  vocabulary is the obvious v2.
- **The correction can only assign a class it was trained on.** If the polygon is truly
  planted pine but no planted pine exists in the padding ring, the model will assign the
  nearest thing it did see. Widen `PAD_M`, or check the §7 class list before believing §10.
- **Single-year features.** Only the target year's embedding is used; the NAIP slider is
  evidence for the analyst, not for the model. Prior-year embeddings and their L2 delta are
  the obvious next feature family — see the leakage discussion in
  `notes/clearcut-vs-agriculture-embeddings.md` before adding them, because features drawn
  from the years that defined the label score near-perfectly by construction.
- **Point-by-point sampling caps the AOI size.** Every apply pixel costs a point in an Earth
  Engine request, and `DEFAULT_MAX_APPLY_PIXELS` (20,000) is where the notebook stops.
  Correcting a county needs an exported embedding raster and a second `FeatureSource`
  implementation — the protocol exists so that swap does not touch anything else.
- **Confidence is a logistic-regression probability**, well calibrated relative to its own
  training distribution and not to ground truth. Treat `MIN_CONFIDENCE` as a control knob to
  be swept, not as a probability of being right.
- **The polygon is an assertion, not a measurement.** Everything downstream inherits whatever
  is wrong about where it was drawn. That is what §5 is for.

**Next steps worth taking:** sweep `MIN_CONFIDENCE` and watch corrected area move; rerun with
`PAD_M` doubled and check the transitions are stable; and compare the corrected raster against
the TreeMap hole strata in `notes/treemap-holes-rectification.md`, which is the same question
asked from the other direction.